In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import joblib
import pandas as pd
import numpy as np

app = Flask(__name__)
CORS(app)

# Load trained model
model = joblib.load("car_price_model.pkl")

# Load scaler
scaler = joblib.load("car_price_scaler.pkl")

# Load feature columns
feature_columns = joblib.load("car_price_features.pkl")


@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Car Price Prediction API is running"
    })


@app.route("/predict", methods=["POST"])
def predict():

    data = request.get_json()

    # Convert input data to DataFrame
    input_data = pd.DataFrame([data])

    # Make sure all required features exist
    for column in feature_columns:
        if column not in input_data.columns:
            input_data[column] = 0

    # Keep exactly the same feature order
    input_data = input_data[feature_columns]

    # Convert to numeric
    input_data = input_data.apply(pd.to_numeric, errors="coerce")
    input_data = input_data.fillna(0)

    # Scale input
    input_scaled = scaler.transform(input_data)

    # Prediction
    prediction = model.predict(input_scaled)

    return jsonify({
        "predicted_price": float(prediction[0])
    })


if __name__ == "__main__":
    app.run(debug=False, port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


In [ ]:
import joblib

feature_columns = joblib.load("car_price_features.pkl")

print("Total features:", len(feature_columns))

for i, col in enumerate(feature_columns, start=1):
    print(i, col)